# Solar Prediction Colab GRU Two-Stage Sweep (Hugging Face)

Multi-seed, resumable GRU sweep on the full dataset for the 4-hour and 24-hour horizons.

1. **Stage 1, inputs:** window length (3 h / 12 h / 24 h) x target-time features (calendar and sun schedule at the target time, plus the target observed whole days before it).
2. **Stage 2, capacity:** hidden size x layers x learning rate on top of each horizon's stage-1 winner.

Configs are ranked by **mean validation RMSE over seeds**. Winners are frozen to CSV before any test metric is computed; then every seed of each winner is scored on the test split together with persistence and seasonal-naive baselines, over all hours and daylight hours only.

Training uses early stopping (`patience`), so `epochs` is only a cap. Each finished config is uploaded to the Hugging Face model repo; rerunning the notebook on a fresh runtime restores finished runs from the Hub and skips them.

Set `HF_TOKEN` in Colab Secrets (`userdata`) or `os.environ['HF_TOKEN']` before running. Use a GPU runtime.

In [ ]:
GIT_REPO_URL = 'https://github.com/itxtx/solar_prediction.git'
PROJECT_DIR = '/content/solar_prediction'

# Hugging Face Hub repositories
HF_DATASET_REPO = 'itxtx/solar-prediction-data'
HF_DATASET_FILENAME = 'solar_weather.csv'
HF_MODEL_REPO = 'itxtx/solar-prediction-artifacts'
HF_PATH_IN_REPO = 'colab_gru_two_stage_sweep'
UPLOAD_TO_HF = True
RESUME_FROM_HF = True

DATA_PATH = 'data/solar_weather.csv'
OUTPUT_ROOT = f'artifacts/{HF_PATH_IN_REPO}'

HORIZONS = {
    '4h': 16,
    '24h': 96,
}
SEEDS = [0, 1, 2]
SEASONAL_LAG = 96  # one day of 15-minute rows

# Shared defaults; epochs is a cap because early stopping uses PATIENCE.
BASE = dict(
    window_size=12,
    target_time_features=False,
    hidden_dim=32,
    num_layers=1,
    learning_rate=1e-3,
    dropout=0.1,
    epochs=100,
    patience=10,
    batch_size=256,
)

# Stage 1: what the model sees.
STAGE1_WINDOWS = [12, 48, 96]
STAGE1_TARGET_TIME = [False, True]

# Stage 2: capacity and optimisation on the stage-1 winner.
STAGE2_HIDDEN = [32, 64]
STAGE2_LAYERS = [1, 2]
STAGE2_LEARNING_RATES = [1e-3, 3e-4]

In [ ]:
import os
import subprocess
from pathlib import Path

if not (Path.cwd() / 'pyproject.toml').exists():
    project = Path(PROJECT_DIR)
    if not project.exists():
        subprocess.run(['git', 'clone', '--depth', '1', GIT_REPO_URL, str(project)], check=True)
    os.chdir(project)

print('Working directory:', Path.cwd())
print('Commit:', subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())

Use a fresh Colab runtime. This notebook installs `huggingface_hub` and the local package without touching Colab's scientific Python stack.

In [ ]:
%pip install -q huggingface_hub
%pip install -q --no-deps -e .

In [ ]:
from solar_prediction.hf_hub import (
    download_artifacts_folder,
    download_dataset_file,
    upload_artifacts_folder,
)

data_path = Path(DATA_PATH)
if not data_path.exists():
    print(f'Downloading {HF_DATASET_FILENAME} from Hugging Face dataset {HF_DATASET_REPO}...')
    data_path = download_dataset_file(
        HF_DATASET_REPO,
        HF_DATASET_FILENAME,
        local_dir=data_path.parent,
    )
assert data_path.exists(), f'Full dataset not found: {data_path}'
print('Full dataset:', data_path)

output_root = Path(OUTPUT_ROOT)
if RESUME_FROM_HF:
    restored = download_artifacts_folder(
        HF_MODEL_REPO, HF_PATH_IN_REPO, local_dir=output_root.parent
    )
    print('Restored finished runs from the Hub:', restored)
output_root.mkdir(parents=True, exist_ok=True)

In [ ]:
import time

import pandas as pd
import torch

from solar_prediction.sweep import (
    RunConfig,
    best_config,
    collect_runs,
    stage1_grid,
    stage2_grid,
    summarize,
    train_run,
    winner_test_metrics,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
base = RunConfig(**BASE)


def upload(message):
    if UPLOAD_TO_HF:
        print('Uploaded:', upload_artifacts_folder(
            output_root, HF_MODEL_REPO, path_in_repo=HF_PATH_IN_REPO, commit_message=message
        ))


def run_stage(configs_by_horizon, stage):
    total = sum(len(configs) for configs in configs_by_horizon.values()) * len(SEEDS)
    done = 0
    for horizon_label, configs in configs_by_horizon.items():
        for config in configs:
            trained_any = False
            for seed in SEEDS:
                done += 1
                started = time.time()
                run_path, trained = train_run(
                    config,
                    horizon_label=horizon_label,
                    horizon_steps=HORIZONS[horizon_label],
                    seed=seed,
                    data_path=data_path,
                    root=output_root,
                    device=device,
                )
                trained_any |= trained
                status = f'trained in {time.time() - started:.0f}s' if trained else 'skipped (already done)'
                print(f'[{stage} {done}/{total}] {horizon_label} {config.name} seed{seed}: {status}')
            if trained_any:
                upload(f'{stage}: {horizon_label} {config.name}')

## Stage 1: window length x target-time features

In [ ]:
stage1 = {h: stage1_grid(base, STAGE1_WINDOWS, STAGE1_TARGET_TIME) for h in HORIZONS}
run_stage(stage1, 'stage1')

runs = collect_runs(output_root)
summary = summarize(runs)
stage1_names = {c.name for configs in stage1.values() for c in configs}
stage1_summary = summary[summary.run_name.isin(stage1_names)]
stage1_summary.to_csv(output_root / 'stage1_summary.csv', index=False)

stage1_winners = {h: best_config(summary, h, stage1[h]) for h in HORIZONS}
for h, config in stage1_winners.items():
    print(h, 'stage-1 winner:', config.name)
stage1_summary[['horizon', 'window_size', 'target_time_features', 'n_seeds',
                'val_rmse_mean', 'val_rmse_std', 'val_r2_mean']]

## Stage 2: hidden size x layers x learning rate on each stage-1 winner

In [ ]:
stage2 = {
    h: stage2_grid(stage1_winners[h], STAGE2_HIDDEN, STAGE2_LAYERS, STAGE2_LEARNING_RATES)
    for h in HORIZONS
}
run_stage(stage2, 'stage2')

runs = collect_runs(output_root)
runs.to_csv(output_root / 'all_runs_validation.csv', index=False)
summary = summarize(runs)
summary.to_csv(output_root / 'all_configs_validation_summary.csv', index=False)
stage2_names = {c.name for configs in stage2.values() for c in configs}
summary[summary.run_name.isin(stage2_names)][
    ['horizon', 'window_size', 'target_time_features', 'hidden_dim', 'num_layers',
     'learning_rate', 'n_seeds', 'val_rmse_mean', 'val_rmse_std', 'val_r2_mean']
]

In [ ]:
import matplotlib.pyplot as plt
from solar_prediction.report_plots import plot_stage2

stage2_fig = plot_stage2(
    summary[summary.run_name.isin(stage2_names)],
    {h: stage1_winners[h].name for h in HORIZONS},
)
stage2_fig.savefig(output_root / 'stage2_validation_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## Freeze winners on validation, then score them once on test

In [ ]:
# Freeze the selection before looking at any test metrics.
final_names = {
    h: best_config(summary, h, stage1[h] + stage2[h]).name for h in HORIZONS
}
best_by_horizon = pd.concat(
    [summary[(summary.horizon == h) & (summary.run_name == name)] for h, name in final_names.items()],
    ignore_index=True,
)
best_by_horizon.to_csv(output_root / 'best_by_horizon.csv', index=False)
best_by_horizon[['horizon', 'run_name', 'n_seeds', 'val_rmse_mean', 'val_rmse_std']]

In [ ]:
test_runs = winner_test_metrics(
    best_by_horizon, runs, data_path, seasonal_lag=SEASONAL_LAG, device=device
)
test_runs.to_csv(output_root / 'final_test_metrics_by_seed.csv', index=False)

test_summary = (
    test_runs.groupby(['horizon', 'horizon_steps', 'run_name', 'scope', 'model'])
    .agg(
        n_seeds=('seed', 'nunique'),
        test_rmse_mean=('test_rmse', 'mean'),
        test_rmse_std=('test_rmse', 'std'),
        test_mae_mean=('test_mae', 'mean'),
        test_r2_mean=('test_r2', 'mean'),
        test_r2_std=('test_r2', 'std'),
    )
    .reset_index()
    .sort_values(['horizon_steps', 'scope', 'test_rmse_mean'])
)
test_summary.to_csv(output_root / 'final_test_metrics_summary.csv', index=False)
test_summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(7 * len(HORIZONS), 4), squeeze=False)
for ax, horizon_label in zip(axes[0], HORIZONS):
    subset = stage1_summary[stage1_summary.horizon == horizon_label]
    for offset, (flag, group) in zip((-0.2, 0.2), subset.groupby('target_time_features')):
        group = group.sort_values('window_size')
        positions = [STAGE1_WINDOWS.index(w) + offset for w in group.window_size]
        ax.bar(positions, group.val_rmse_mean, width=0.4, yerr=group.val_rmse_std.fillna(0),
               capsize=3, label='target-time features' if flag else 'no target-time features')
    ax.set_xticks(range(len(STAGE1_WINDOWS)))
    ax.set_xticklabels([f'{w} rows ({w / 4:g} h)' for w in STAGE1_WINDOWS])
    ax.set_title(f'Stage 1 validation RMSE, {horizon_label} horizon (mean +/- std over seeds)')
    ax.set_ylabel('W/m^2')
    lowest = subset.val_rmse_mean.min()
    ax.set_ylim(bottom=max(0, lowest - 3 * subset.val_rmse_std.fillna(0).max() - 1))
    ax.legend()
plt.tight_layout()
fig.savefig(output_root / 'stage1_validation_plot.png', dpi=150)
plt.show()

upload('Two-stage sweep summaries, frozen winners, and test metrics')